<a href="https://colab.research.google.com/github/rupikap009/ML-Techniques/blob/main/notebooks/01_house_rent_regression.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# House Rent Prediction with Linear Regression

This notebook evaluates an interpretable OLS baseline for monthly rent. The target is modeled as `log1p(Rent)` to reduce right skew, while final errors are reported in the original currency units.

**Research question.** How accurately can basic property and tenant-preference variables predict monthly rent, and how does OLS compare with a no-skill baseline and Ridge regression?

## 1. Data source and reproducible loading

The notebook uses `House_Rent_Dataset.csv` from the [House Rent Prediction Dataset on Kaggle](https://www.kaggle.com/datasets/iamsouravbanerjee/house-rent-prediction-dataset). Because the repository does not include the data, record the exact dataset version and access date, and place the CSV beside this notebook or in `/content/` on Colab. Do not claim that the dataset is nationally representative.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.model_selection import KFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
import scipy.stats as stats

sns.set_theme(style="whitegrid")
DATA_FILENAME = "House_Rent_Dataset.csv"
candidates = [Path(DATA_FILENAME), Path("data") / DATA_FILENAME, Path("/content") / DATA_FILENAME]
dataset_path = next((p for p in candidates if p.exists()), None)
if dataset_path is None:
    raise FileNotFoundError("Download House_Rent_Dataset.csv and place it beside this notebook, in data/, or in /content/.")
df = pd.read_csv(dataset_path)
required = {"Rent", "BHK", "Size", "Area Type", "City", "Furnishing Status", "Tenant Preferred", "Bathroom", "Point of Contact"}
missing = required - set(df.columns)
if missing:
    raise KeyError(f"Missing required columns: {sorted(missing)}")
print(f"Loaded {dataset_path}: {df.shape[0]:,} rows x {df.shape[1]} columns")
print(df.isna().sum())
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(df["Rent"], kde=True, ax=axes[0]); axes[0].set_title("Rent")
sns.histplot(np.log1p(df["Rent"]), kde=True, ax=axes[1]); axes[1].set_title("log1p(Rent)")
plt.tight_layout(); plt.show()

## 2. Feature decisions and preprocessing

`Posted On` is excluded because it is a collection/listing date rather than a property characteristic; `Area Locality` and raw `Floor` are excluded from this compact baseline because locality is high-cardinality and floor requires domain-specific parsing. These exclusions are modeling choices, not proof that the variables are irrelevant. A later study should test them explicitly.

The `ColumnTransformer` is fitted only on training data, preventing category vocabulary and imputation statistics from entering the test set.

In [ ]:
target = np.log1p(df["Rent"].astype(float))
feature_cols = ["BHK", "Size", "Area Type", "City", "Furnishing Status", "Tenant Preferred", "Bathroom", "Point of Contact"]
X = df[feature_cols].copy()
y = target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

numeric_features = ["BHK", "Size", "Bathroom"]
categorical_features = [c for c in feature_cols if c not in numeric_features]
try:
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse_output=False)
except TypeError:  # compatibility with older scikit-learn
    encoder = OneHotEncoder(handle_unknown="ignore", drop="first", sparse=False)
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")), ("scaler", StandardScaler())]), numeric_features),
    ("categorical", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("onehot", encoder)]), categorical_features),
])
ols_model = Pipeline([("preprocess", preprocessor), ("regressor", LinearRegression())])
ridge_model = Pipeline([("preprocess", preprocessor), ("regressor", Ridge(alpha=1.0))])
baseline_model = Pipeline([("preprocess", preprocessor), ("regressor", DummyRegressor(strategy="median"))])

## 3. Baselines and cross-validation

Five-fold cross-validation is performed on `log1p(Rent)`. The median dummy model is a no-skill reference; Ridge tests whether mild regularization improves the one-hot linear model. The final holdout is retained for a single untouched evaluation.

In [ ]:
def rmse(y_true, y_pred):
    return np.sqrt(mean_squared_error(y_true, y_pred))
scoring = {"r2": "r2", "mae_log": "neg_mean_absolute_error", "rmse_log": make_scorer(rmse, greater_is_better=False)}
cv = KFold(n_splits=5, shuffle=True, random_state=42)
for name, estimator in [("Median baseline", baseline_model), ("OLS", ols_model), ("Ridge", ridge_model)]:
    result = cross_validate(estimator, X, y, cv=cv, scoring=scoring, n_jobs=-1)
    print(f"\n{name} (5-fold CV on log rent)")
    print(f"R2:       {result['test_r2'].mean():.4f} +/- {result['test_r2'].std():.4f}")
    print(f"MAE log:  {-result['test_mae_log'].mean():.4f} +/- {result['test_mae_log'].std():.4f}")
    print(f"RMSE log: {-result['test_rmse_log'].mean():.4f} +/- {result['test_rmse_log'].std():.4f}")

ols_model.fit(X_train, y_train)
y_pred_log = ols_model.predict(X_test)
y_pred_actual = np.maximum(0, np.expm1(y_pred_log))
y_test_actual = np.expm1(y_test)
print("\nUntouched holdout (original rent units)")
print(f"R2:   {r2_score(y_test_actual, y_pred_actual):.4f}")
print(f"MAE:  {mean_absolute_error(y_test_actual, y_pred_actual):.2f}")
print(f"RMSE: {rmse(y_test_actual, y_pred_actual):.2f}")

In [ ]:
residuals = y_test - y_pred_log
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].scatter(y_pred_log, residuals, alpha=0.5)
axes[0].axhline(0, color="red", linestyle="--")
axes[0].set(xlabel="Fitted log rent", ylabel="Residual", title="Residuals vs fitted")
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title("Normal Q-Q plot")
plt.tight_layout(); plt.show()

print("Residual diagnostics are descriptive; they do not prove all OLS assumptions.")

## 4. Limitations, citations, and reproduction

- The random split estimates performance for observations similar to this dataset, not future market conditions or unseen cities.
- The dataset is relatively small and may contain listing- or sampling-specific patterns; results should not be generalized beyond its source population.
- Log-target metrics and original-rent metrics answer different questions; both are reported explicitly. Exponentiating predictions can introduce retransformation bias, so the reported original-scale results should be interpreted as a baseline rather than unbiased conditional means.
- Feature exclusions are deliberate baseline choices and should be compared with a locality/floor-enhanced model in future work.
- To reproduce: obtain the exact `House_Rent_Dataset.csv` from the dataset source cited in the report, place it beside this notebook (or `/content/` on Colab), and run all cells.

**Citation:** iamsouravbanerjee, *House Rent Prediction Dataset*, Kaggle, https://www.kaggle.com/datasets/iamsouravbanerjee/house-rent-prediction-dataset. Include the dataset version and access date; the filename alone is not a sufficient academic citation.